In [ ]:
# Target Destination for the CSV output
OUTPUT_CSV = "../2.model_training/soulsync_raw_data.csv"

# Upper-body landmark indices (MediaPipe Pose)
# MediaPipe Pose has 33 landmarks, but we focus on a subset relevant to posture analysis
TRACKED_LANDMARKS = {
    "nose":            0,
    "left_shoulder":   11,
    "right_shoulder":  12,
    "left_elbow":      13,
    "right_elbow":     14,
    "left_wrist":      15,
    "right_wrist":     16,
    "left_hip":        23,
    "right_hip":       24}						    

HEADER = []       # CSV header will be dynamically generated based on tracked landmarks

for name in TRACKED_LANDMARKS:
    # Each landmark contributes 4 columns: x, y, z, visibility
    HEADER += [f"{name}_x", f"{name}_y", f"{name}_z", f"{name}_visibility"]

# Additional features for wrist-nose distance and hand proximity    
HEADER += ["left_wrist_nose_dist", "right_wrist_nose_dist", "hands_near_face", "label"]

In [22]:
import math     # Math module for distance calculation


# function to calculate Simple Euclidean distance between two landmarks
def euclidean(p1, p2): 
    return math.sqrt((p1.x - p2.x) ** 2 + (p1.y - p2.y) ** 2)

In [23]:
WRIST_NOSE_THRESHOLD = 0.25    # Threshold for considering wrists near the face (tuned empirically)


# function to extract features from landmarks and return a row for CSV 
def extract_row(landmarks, label):
    lm = landmarks.landmark     # list of 33 landmarks
    row = []                    # Start with an empty row and append features for each tracked landmark
    
    for index in TRACKED_LANDMARKS.values():   # Loop through the indices of the tracked landmarks
        pt = lm[index]   					   # Get the landmark point
        
        # Append the x, y, z coordinates and visibility of the landmark to the row
        row += [round(pt.x, 6), round(pt.y, 6), round(pt.z, 6), round(pt.visibility, 4)]
        
	# Calculate wrist-nose distances and hand proximity features	
    nose, lw, rw = lm[0], lm[15], lm[16]
    l_dist = round(euclidean(lw, nose), 6)   # Calculate distance from left wrist to nose
    r_dist = round(euclidean(rw, nose), 6)   # Calculate distance from right wrist to nose
    
    # Determine if hands are near the face based on the defined threshold
    near = int(l_dist < WRIST_NOSE_THRESHOLD or r_dist < WRIST_NOSE_THRESHOLD) # int(true) = 1
    row += [l_dist, r_dist, near, label]                                       # int(false) = 0
    return row

In [24]:
import cv2      # OpenCV for video capture and drawing

# Function to draw the overlay on the video frame with recording status, label, and row count
def draw_overlay(frame, recording, label, row_count):
    
    h, w = frame.shape[:2]      # Get the height and width of the frame 
           
    color = (0, 200, 80) if recording else (60, 60, 60)      
    status = "o REC" if recording else "* PAUSED"       
    label_names = {0: "Balanced", 1: "Slumped/Exhausted", 2: "Restless/Fidgeting"}
    
    
    cv2.rectangle(frame, (0, 0), (w, 90), (20, 20, 20), -1)  # Background for text
    
	# Display the recording status at the top-left corner of the frame
    cv2.putText(frame, status, (16, 34), cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2)
	

	# Display the current label and its name, positioned below the recording status
    cv2.putText(frame, f"Label: [{label}] {label_names.get(label, '?')}",
                (16, 62), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (200, 200, 200), 1)
    

	# Display the count of rows saved to the CSV file, positioned below the label information
    cv2.putText(frame, f"Rows saved: {row_count}", 
                (16, 82), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (140, 140, 140), 1)
    

    # Instructions for user controls, displayed at the bottom of the frame
    cv2.putText(frame, "0/1/2=label  R=rec  Q=quit",
                (16, h - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 0, 0), 1)

print("Helpers & environment configuration cells compiled successfully!")

Helpers & environment configuration cells compiled successfully!


In [25]:
import mediapipe as mp    # MediaPipe for pose estimation
import csv                # CSV module for writing data
import os                 # OS module for file handling


# Main function to run the data collection process

def main():      # "../2.model_training"
    os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)  # Ensure the output directory exists
    
	# check if the output CSV file already exists to determine if we need to write the header
    file_exists = os.path.isfile(OUTPUT_CSV)            

    mp_pose = mp.solutions.pose            # Initialize MediaPipe Pose solution for pose estimation
    mp_draw = mp.solutions.drawing_utils   # and drawing utilities for visualizing landmarks

    cap = cv2.VideoCapture(0)        # Start video capture from the default webcam
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)    # Set the frame width to 640 pixels
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 460)   # Set the frame height to 460q pixels

    # Initialize variables for recording state, label, and row count
    recording  = False          # no recording  
    label      = 0              # balanced
    row_count  = 0              # no samples collected

    # Open the CSV file in append mode and create a CSV writer object
    csv_file   = open(OUTPUT_CSV, "a", newline="")  
    writer     = csv.writer(csv_file)
    
	# If the file didn't exist before, write the header row to the CSV file
    if not file_exists:
        writer.writerow(HEADER)
    

    print("SoulSync data collector ready.")
    print("Press 0/1/2 to set label, R to toggle recording, Q to quit.\n")


    # Use MediaPipe Pose in a context manager to ensure proper resource management
    with mp_pose.Pose(min_detection_confidence=0.6,
                min_tracking_confidence=0.6,model_complexity=1) as pose:
        
		# Main loop to read frames from the webcam, process them, and handle user input
        while cap.isOpened():        # while the webcam is open and available
            ret, frame = cap.read()  # Read a frame from the webcam
            if not ret:              # If frame reading fails, exit the loop
                break

            frame = cv2.flip(frame, 1)   # Flip the frame horizontally for a mirror-like view
            # Convert the frame from BGR to RGB color space for MediaPipe processing
            rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = pose.process(rgb)  # Process the frame with MediaPipe Pose to get landmark results


            # If pose landmarks are detected, 
			# draw them on the frame and extract features if recording is active
            if results.pose_landmarks:
                
                mp_draw.draw_landmarks(frame, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                            mp_draw.DrawingSpec(color=(80, 200, 120), thickness=2, circle_radius=3),
                            mp_draw.DrawingSpec(color=(80, 80, 80),  thickness=1),)
                
                # If recording is active, 
				# extract the features from the detected landmarks and write them to the CSV file
                if recording:
                    row = extract_row(results.pose_landmarks, label)
                    writer.writerow(row)
                    row_count += 1
            

			# Draw the overlay with recording status, label, and row count on the frame,
			# then display it in a window
            draw_overlay(frame, recording, label, row_count)
            cv2.imshow("SoulSync — Data Collection", frame)
            

			# Handle user key presses for quitting, toggling recording, and changing labels
            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break
            elif key == ord("r"):
                recording = not recording
                state = "STARTED" if recording else "PAUSED"
                print(f"Recording {state} — label={label}, rows so far={row_count}")
            elif key in (ord("0"), ord("1"), ord("2")):
                label = int(chr(key))  
                print(f"Label set to {label}")

    # Cleanup: close the CSV file, release the webcam, and close OpenCV windows
    csv_file.close()
    cap.release()
    cv2.destroyAllWindows()
    print(f"\nDone. {row_count} rows saved to:\n  {os.path.abspath(OUTPUT_CSV)}")

if __name__ == "__main__":
    main()

SoulSync data collector ready.
Press 0/1/2 to set label, R to toggle recording, Q to quit.

Recording STARTED — label=0, rows so far=0
Label set to 1
Label set to 2
Label set to 2

Done. 6094 rows saved to:
  h:\SoulSync\2.model_training\soulsync_raw_data.csv
